In [40]:
import pandas as pd
import random
from datetime import datetime, timedelta

def generate_schedule(year=2026):
    # Создаем диапазон дат на весь год
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    all_days = pd.date_range(start_date, end_date)

    schedule_data = []

    for day in all_days:
        # Пн-Сб (0-5), Воскресенье (6) пропускаем
        if day.weekday() < 6:
            date_str = day.strftime('%Y-%m-%d')
            day_name = day.strftime('%A')

            # 1. Начало дня всегда в 10:00
            work_start = datetime.strptime("10:00", "%H:%M")
            
            # 2. Рандомим начало перерыва (от 14:00 до 15:00)
            # Можно выбрать шаг, например, каждые 5 минут, чтобы график не был слишком "рваным"
            random_minutes = random.randrange(0, 61, 60) 
            break_start = datetime.strptime("14:00", "%H:%M") + timedelta(minutes=random_minutes)
            
            # 3. Конец перерыва ВСЕГДА через 3 часа
            break_end = break_start + timedelta(hours=3)
            
            # 4. Конец рабочего дня всегда в 23:00
            work_end = datetime.strptime("23:00", "%H:%M")

            # Считаем часы (всегда будет 10 часов работы, так как 13 часов всего минус 3 часа перерыва)
            total_work_time = (break_start - work_start) + (work_end - break_end)
            total_hours = total_work_time.total_seconds() / 3600

            schedule_data.append({
                'Дата': date_str,
                'День недели': day_name,
                'Начало работы': work_start.strftime("%H:%M"),
                'Уход на перерыв': break_start.strftime("%H:%M"),
                'Возврат с перерыва': break_end.strftime("%H:%M"),
                'Конец дня': work_end.strftime("%H:%M"),
                'Длительность перерыва': "3 часа",
                'Всего раб. часов': int(total_hours)
            })

    # Создаем таблицу
    df = pd.DataFrame(schedule_data)
    
    # Сохраняем результаты
    file_name = f'Work_Schedule_3h_Break_{year}'
    df.columns = ['date','dw','work_start','break_start','break_end','work_end','break_duration', 'working_hours']
    df.to_excel(f'{file_name}.xlsx', index=False)
    df.to_csv(f'{file_name}.csv', index=False, sep=';', encoding='utf-8')
    
    print(f"График на {year} год готов!")
    print(f"Файлы: {file_name}.xlsx и {file_name}.csv")
    return df

# Запуск
df_schedule = generate_schedule(2026)




График на 2026 год готов!
Файлы: Work_Schedule_3h_Break_2026.xlsx и Work_Schedule_3h_Break_2026.csv


In [ ]:
import asyncio
import logging
import os
import psycopg
import sqlalchemy

from datetime import datetime
from typing import Type
from sqlalchemy import Table, Column, Integer, String, text
from sqlalchemy.dialects.postgresql import insert
from sqlmodel import create_engine, SQLModel, Field, MetaData, select
from sqlmodel.ext.asyncio.session import AsyncSession
from sqlalchemy.ext.asyncio import create_async_engine
from typing import Union, Dict, Iterable
from pathlib import Path
from pydantic import conset

import keys as const


if os.name == 'nt':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())


async def is_alive(conn):
    if conn is None or conn.closed:
        return False
    try:
        await conn.set_autocommit(False)
        cur = conn.cursor()
        await cur.execute("CREATE TEMP TABLE tmp_write_test(id INT);")
        await cur.execute("INSERT INTO tmp_write_test VALUES (1);")
        await conn.rollback()
        await cur.close()
        return True
    except (psycopg.OperationalError, psycopg.ProgrammingError, psycopg.errors.ReadOnlySqlTransaction) as e:
        return False


def keys(db_user=None, db_pass=None, db_host=None):
    if db_host is None:
        db_host = const.DB_HOST
        # db_host = os.getenv("DB_HOST")
    if db_user is None:
        db_user = const.LOG
        # db_user = os.getenv("DB_USER")
    if db_pass is None:
        db_pass = const.PASS
        # db_pass = os.getenv("DB_PASS")
    return db_host, db_user, db_pass


def create_temp_table(engine):
    """Создает временную таблицу."""
    metadata = MetaData()
    temp_table = Table(
        "temp_table",
        metadata,
        Column("id", Integer, primary_key=True),
        Column("value", String),
    )

    try:
        with engine.connect() as connection:
            temp_table.create(bind=engine)
            return temp_table
    except (psycopg.OperationalError, psycopg.ProgrammingError, psycopg.errors.ReadOnlySqlTransaction,
            sqlalchemy.exc.InternalError) as e:
        return None


def is_read_only(engine):
    temp_table = create_temp_table(engine)  # Создаем таблицу здесь
    if temp_table is None:
        return True  # Если не удалось создать таблицу, считаем хост только для чтения

    try:
        with engine.connect() as connection:
            connection.execute(temp_table.insert().values(value="test data"))
            return False  # Запись прошла успешно - хост для записи
    except Exception as e:
        return True  # Ошибка записи - хост только для чтения
    finally:
        # Убеждаемся, что таблица удалена
        try:
            with engine.connect() as connection:
                temp_table.drop(bind=engine)
        except Exception as e:
            print(f"Ошибка при удалении временной таблицы: {e}")


def get_engine(*args, **kwargs, ):
    db_host, db_user, db_pass = keys(*args, **kwargs)
    db_port = const.DB_PORT
    db_schema = const.DB_DATABASE
    hosts = db_host.split(',')
    # url = f'postgresql+asyncpg://{db_user}:{db_pass}@{db_host}:{db_port}/{db_schema}'
    for host in hosts:
        # url = 'postgresql+psycopg://{u}:{p}@{h}:{port}/cpu'.format(
        #     u=db_user, p=db_pass, h=host, port='5432')
        url = f'postgresql+psycopg://{db_user}:{db_pass}@{host}:{db_port}/{db_schema}'
        if is_read_only(create_engine(url)):
            continue
        url = f'postgresql+asyncpg://{db_user}:{db_pass}@{host}:{db_port}/{db_schema}'
        # url = 'postgresql+asyncpg://{u}:{p}@{h}:{port}/cpu'.format(
        #     u=db_user, p=db_pass, h=host, port='5432')
        break
    else:
        raise Exception("Не удалось подключиться ни к одному из хостов")

    return create_async_engine(url,
                               pool_pre_ping=True,
                               pool_size=10,
                               max_overflow=20,
                               pool_recycle=1800, )


def get_sync_engine(*args, **kwargs, ):
    db_host, db_user, db_pass = keys(*args, **kwargs)
    db_port = const.DB_PORT
    db_schema = const.DB_DATABASE
    db_host, *_ = db_host.split(',')
    url = f'postgresql+psycopg://{db_user}:{db_pass}@{db_host}:{db_port}/{db_schema}'

    return create_engine(url)


engine = get_engine()
class Base(SQLModel):
    metadata = MetaData(schema=const.DB_DATABASE)


class Products(Base, table=True):
    __tablename__ = 'products'
    id: int = Field(default=None, primary_key=True)
    product_id: str = Field(default=None)
    name: str = Field(default=None)
    description: str = Field(default=None)
    price: int = Field(default=None)
    number: str = Field(default=None)
    time_frame: datetime = Field(default=None)
    status: str = Field(default=None)


async def all_item(table: Type[Base], lim: int = None):
    """
    Достать несколько элементов из таблицы, по умолчанию - все
    """
    async with AsyncSession(engine) as session:
        statement = select(table)
        if lim:
            statement = statement.limit(lim)
        results = await session.exec(statement)
        result = results.all()
        return result
    
df = await all_item(Products, 300)

OperationalError: (psycopg.OperationalError) connection failed: connection to server at "193.233.216.12", port 5432 failed: FATAL:  database "public" does not exist
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [9]:
pip install greenlet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
